# Brain Tumour Segmentation on MRI (BRISC 2025) — Full Pipeline, Validation & Analysis

**SDAIA Academy · Computer Vision Systems Development · Final Project**

This notebook reproduces every step of the project, from environment check to final analysis,
and adds validation checks and in-depth evaluation plots. It drives the tested scripts in `src/`
(so the notebook and the command line always produce identical results) and analyses their outputs.

| Step | Content | Project requirement |
|---|---|---|
| 0 | Setup & run switches | — |
| 1 | Environment check (Apple M4, MPS) | Deployment context |
| 2 | Data download, indexing, splits + **data validation** | 4. Data & Model |
| 3 | Exploratory data analysis | 4. Data & Model |
| 4 | Pipeline sanity checks: dataset, augmentation, models, loss, metrics | 3. CV Implementation |
| 5 | Smoke test on synthetic data (optional) | 3. CV Implementation |
| 6 | Training + **training analysis** | 3–4 |
| 7 | Test-set evaluation + **result validation** + model comparison | 5. Evaluation |
| 8 | Statistical analysis: confidence intervals, paired tests | 5. Evaluation |
| 9 | Healthy slices & image-level tumour detection (ROC) | 5. Evaluation |
| 10 | Improvement experiments: post-processing, TTA, threshold tuning | 5. Evaluation |
| 11 | Qualitative results: successes & failure cases | 5. Evaluation |
| 12 | Deployment: ONNX export, INT8 quantization, benchmark, demo app | 6. Deployment |
| 13 | Report generation, summary of findings & validation report | 7. Submission |

**Model**

The submitted model is `unet_resnet34`: a U-Net with an ImageNet-pretrained ResNet34 encoder (24.4 M parameters),
trained on the tumour slices. Configurations for three further variants are included in `configs/`
(`unet_scratch`, `unet_effb0`, `unet_resnet34_healthy`) and can be added to `VARIANTS` in Step 0 once trained;
every analysis below then compares all available runs automatically.

### How to use this notebook

1. Register the project environment as a Jupyter kernel once (in Terminal):
   `conda activate brisc-seg && python -m ipykernel install --user --name brisc-seg --display-name "Python (brisc-seg)"`,
   then choose **Python (brisc-seg)** as this notebook's kernel.
2. Set the switches in **Step 0**. Completed steps are detected from their output files and skipped,
   so re-running the notebook never retrains or re-evaluates by accident.
3. **Long training runs (~2 h) are best started in Terminal** (`python -m src.train --config ...`):
   closing the notebook or a kernel restart would kill a training run started from here.
   The notebook picks up finished runs automatically.
4. Keep the MacBook plugged in and on a hard surface during training (fanless → thermal throttling).

---
## Step 0 · Setup and run switches

In [ ]:
import os, sys, json, time, itertools, subprocess, warnings
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")
os.environ.setdefault("NO_ALBUMENTATIONS_UPDATE", "1")

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yaml
from IPython.display import Image, Markdown, display

pd.set_option("display.precision", 4)
pd.set_option("display.max_columns", 30)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})
warnings.filterwarnings("ignore", category=UserWarning)

# ---------------- paths ----------------
DATA_ROOT = ROOT / os.environ.get("BRISC_DATA_ROOT", "data/raw")
SPLITS    = ROOT / os.environ.get("BRISC_SPLITS_DIR", "data/splits")
RUNS      = ROOT / os.environ.get("BRISC_RUNS_DIR", "runs")
ASSETS    = ROOT / "assets"
CACHE     = ROOT / "data" / "cache"
for p in (ASSETS, CACHE):
    p.mkdir(parents=True, exist_ok=True)

# ---------------- run switches ----------------
SKIP_COMPLETED           = True   # never redo a step whose outputs already exist
RUN_ENV_CHECK            = True   # Step 1 (≈ 30 s)
RUN_DOWNLOAD             = False  # Step 2: force re-download (automatic if data is missing)
RUN_PREPARE              = False  # Step 2: force re-indexing (automatic if split CSVs are missing)
RUN_DUPLICATE_CHECK      = True   # Step 2: perceptual-hash leakage check (≈ 1 min)
RUN_MODEL_CHECKS         = True   # Step 4 (≈ 30 s)
RUN_SMOKE_TEST           = False  # Step 5 (≈ 2 min)
RUN_TRAINING             = False  # Step 6: train missing variants FROM THE NOTEBOOK (hours!)
RUN_EVALUATION           = True   # Step 7: evaluate trained runs that have no evaluation yet
RUN_POSTPROC_EXPERIMENTS = True   # Step 10: min-area / TTA re-evaluations (a few min per run)
RUN_THRESHOLD_SWEEP      = True   # Step 10: validation-set threshold sweep (≈ 1–2 min per run)
RUN_TUNED_THRESHOLD_EVAL = True   # Step 10: re-evaluate the test set at the tuned threshold
RUN_CROSS_MODEL_GALLERY  = True   # Step 11: hardest cases (all available models side by side)
RUN_DEPLOYMENT           = True   # Step 12: ONNX export, INT8 quantization, benchmark (≈ 5–10 min)
RUN_REPORT               = True   # Step 13: diagrams + README results section

# ---------------- experiment definition ----------------
VARIANTS = {
    "unet_resnet34":         "configs/unet_resnet34.yaml",      # submitted model
    # "unet_scratch":        "configs/unet_scratch.yaml",       # optional extra variants:
    # "unet_effb0":          "configs/unet_effb0.yaml",         # uncomment after training them
    # "unet_resnet34_healthy": "configs/unet_resnet34_healthy.yaml",
}
LABELS = {
    "unet_scratch":          "U-Net (scratch)",
    "unet_resnet34":         "U-Net + ResNet34",
    "unet_effb0":            "U-Net + EffNet-B0",
    "unet_resnet34_healthy": "U-Net + ResNet34 + healthy",
}
BASELINE_RUN = "unet_resnet34"        # reference for paired comparisons
HEALTHY_RUN  = "unet_resnet34_healthy"
SEED = 42
COLORS = dict(zip(LABELS, plt.rcParams["axes.prop_cycle"].by_key()["color"]))

print("Project root:", ROOT)
print("Data root   :", DATA_ROOT)

In [ ]:
# ---------------- helpers ----------------
def run(args, python=True):
    # Run a project command, stream its output live, raise if it fails.
    cmd = [sys.executable, *args] if python else list(args)
    print("$", " ".join((["python"] if python else []) + [str(a) for a in (args if python else args)]))
    env = {**os.environ, "TQDM_DISABLE": "1", "PYTHONUNBUFFERED": "1"}
    t0 = time.time()
    with subprocess.Popen(cmd, cwd=ROOT, env=env, stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True, bufsize=1) as proc:
        for line in proc.stdout:
            print(line, end="")
    if proc.returncode != 0:
        raise RuntimeError(f"Command failed with exit code {proc.returncode}")
    print(f"[finished in {(time.time() - t0) / 60:.1f} min]\n")


CHECKS = []
def check(name, ok, detail=""):
    # Record a validation check; all checks are summarised in Step 12.
    CHECKS.append({"check": name, "passed": bool(ok), "detail": detail})
    print(("✅" if ok else "⚠️ "), name, f"— {detail}" if detail else "")


def savefig(name):
    plt.savefig(ASSETS / f"analysis_{name}.png", dpi=150, bbox_inches="tight")


def label(run_name):
    return LABELS.get(run_name, run_name)

---
## Step 1 · Environment check

Confirms package versions, that PyTorch can use the Apple GPU through **Metal Performance Shaders (MPS)**,
and benchmarks one training step. On the M4 MacBook Air (16 GB unified memory) this measured
≈ 0.33 s per step (batch 8, 256 px) → ≈ 24 images/s for the scratch U-Net.

In [ ]:
if RUN_ENV_CHECK:
    run(["scripts/check_env.py"])

import torch
from src.utils import get_device, load_config, set_seed

DEVICE = get_device()
set_seed(SEED)
check("PyTorch MPS backend available", torch.backends.mps.is_available(), f"device = {DEVICE}")

---
## Step 2 · Data: download, indexing, splits and validation

**Dataset — BRISC 2025** ([Kaggle](https://www.kaggle.com/datasets/briscdataset/brisc2025),
[paper](https://arxiv.org/abs/2506.14318), CC BY 4.0): 6,000 contrast-enhanced T1-weighted MRI slices
(512×512) annotated by radiologists and physicians, in axial, coronal and sagittal planes.

Important property discovered while downloading: the **segmentation task contains tumour slices only**
(3,933 train / 860 test = exactly the glioma + meningioma + pituitary counts of the classification task).
Healthy slices exist only in `classification_task/*/no_tumor` (1,067 train / 140 test). We therefore:

* keep the official test split untouched as the benchmark,
* carve a **stratified validation set** (15 %, by tumour type × plane) out of the training split,
* index healthy slices separately (empty masks) to measure **false positives** and for the improvement experiment.

`src.prepare_data` writes fixed split CSVs (`data/splits/`), which are committed for reproducibility.

In [ ]:
def data_present():
    return DATA_ROOT.exists() and any(p.name == "segmentation_task" for p in DATA_ROOT.rglob("*") if p.is_dir())

if RUN_DOWNLOAD or not data_present():
    run(["bash", "scripts/download_data.sh"], python=False)
else:
    print("Dataset already present in", DATA_ROOT)

if RUN_PREPARE or not (SPLITS / "train.csv").exists() or not (SPLITS / "healthy_test.csv").exists():
    run(["-m", "src.prepare_data", "--data-root", str(DATA_ROOT.relative_to(ROOT)),
         "--out", str(SPLITS.relative_to(ROOT)), "--seed", str(SEED)])
else:
    print("Split CSVs already present in", SPLITS)

SPLIT_NAMES = ["train", "val", "test", "healthy_train", "healthy_test"]
splits = {n: pd.read_csv(SPLITS / f"{n}.csv").fillna({"mask": ""}) for n in SPLIT_NAMES}
split_summary = json.loads((SPLITS / "summary.json").read_text())
pd.DataFrame({n: [len(df)] for n, df in splits.items()}, index=["images"])

### 2.1 Data validation

Automated checks before any training: expected sizes, parsed metadata, file existence,
mask encoding, stratification quality and **no identical file between splits**.

In [ ]:
from src.data_index import read_mask, read_image_rgb, load_pair

EXPECTED = {"train": 3343, "val": 590, "test": 860, "healthy_train": 1067, "healthy_test": 140}
for n, expected in EXPECTED.items():
    check(f"split size: {n}", len(splits[n]) == expected, f"{len(splits[n])} (expected {expected})")

tumour_df = pd.concat([splits["train"], splits["val"], splits["test"]], ignore_index=True)
check("tumour type parsed for every image", (tumour_df["tumor_type"] != "unknown").all(),
      f"{(tumour_df['tumor_type'] == 'unknown').sum()} unknown")
check("imaging plane parsed for every image", (tumour_df["plane"] != "unknown").all(),
      f"{(tumour_df['plane'] == 'unknown').sum()} unknown")

missing = [p for df in splits.values() for col in ("image", "mask") for p in df[col] if p and not (DATA_ROOT / p).exists()]
check("all referenced files exist", not missing, f"{len(missing)} missing")

stems = {n: set(df["stem"]) for n, df in splits.items()}
for a, b in itertools.combinations(SPLIT_NAMES, 2):
    overlap = stems[a] & stems[b]
    if overlap or {a, b} & {"test", "healthy_test"}:
        check(f"no file overlap {a} ↔ {b}", not overlap, f"{len(overlap)} shared")

check("healthy slices have no mask file", (splits["healthy_train"]["mask"] == "").all() and (splits["healthy_test"]["mask"] == "").all())

# mask encoding on a random sample
sample = tumour_df.sample(n=min(300, len(tumour_df)), random_state=SEED)
raw_values, empty, shape_mismatch = set(), 0, 0
for _, r in sample.iterrows():
    raw = cv2.imread(str(DATA_ROOT / r["mask"]), cv2.IMREAD_GRAYSCALE)
    img = cv2.imread(str(DATA_ROOT / r["image"]), cv2.IMREAD_GRAYSCALE)
    raw_values.update(np.unique(raw).tolist())
    empty += int(read_mask(DATA_ROOT / r["mask"]).sum() == 0)
    shape_mismatch += int(raw.shape != img.shape)
check("tumour masks are non-empty (sample)", empty == 0, f"{empty}/{len(sample)} empty")
check("mask size matches image size (sample)", shape_mismatch == 0, f"{shape_mismatch} mismatches")
print("Distinct raw mask values in sample:", sorted(raw_values)[:12], "..." if len(raw_values) > 12 else "")

# stratification quality: class/plane proportions train vs val
strata = lambda df: (df["tumor_type"] + "|" + df["plane"]).value_counts(normalize=True)
max_diff = (strata(splits["train"]) - strata(splits["val"])).abs().max()
check("train/val strata proportions match", max_diff < 0.02, f"max difference {max_diff:.3%}")

### 2.2 Near-duplicate check (perceptual hashing)

File names can differ while pixel content is (almost) identical, which would leak test information into training.
A 64-bit **difference hash** (dHash) is computed for every slice; test slices whose hash also occurs on the
training side are listed as *candidates* and shown for visual inspection (similar-looking slices can collide).

In [ ]:
def dhash(path, size=8):
    img = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    small = cv2.resize(img, (size + 1, size), interpolation=cv2.INTER_AREA)
    return np.packbits(small[:, 1:] > small[:, :-1]).tobytes().hex()

if RUN_DUPLICATE_CHECK:
    hash_cache = CACHE / "dhash.csv"
    if hash_cache.exists() and SKIP_COMPLETED:
        hashes = pd.read_csv(hash_cache)
    else:
        rows = [{"split": n, "stem": r["stem"], "image": r["image"], "hash": dhash(DATA_ROOT / r["image"])}
                for n, df in splits.items() for _, r in df.iterrows()]
        hashes = pd.DataFrame(rows)
        hashes.to_csv(hash_cache, index=False)

    train_side = hashes[hashes["split"].isin(["train", "val", "healthy_train"])]
    test_side = hashes[hashes["split"].isin(["test", "healthy_test"])]
    dup = test_side.merge(train_side, on="hash", suffixes=("_test", "_train"))
    within_test = test_side["hash"].duplicated().sum()
    check("no near-duplicate test ↔ train slices (dHash)", dup.empty,
          f"{dup['stem_test'].nunique()} candidate test slices out of {len(test_side)}")
    print(f"Duplicate hashes inside the test side: {within_test}")

    if not dup.empty:
        show = dup.drop_duplicates("stem_test").head(4)
        fig, axes = plt.subplots(2, len(show), figsize=(3 * len(show), 6), squeeze=False)
        for j, (_, r) in enumerate(show.iterrows()):
            for i, (key, title) in enumerate([("image_test", "test"), ("image_train", "train side")]):
                axes[i, j].imshow(cv2.imread(str(DATA_ROOT / r[key]), cv2.IMREAD_GRAYSCALE), cmap="gray")
                axes[i, j].set_title(f"{title}\n{Path(r[key]).stem[:28]}", fontsize=7)
                axes[i, j].axis("off")
        plt.suptitle("Near-duplicate candidates (inspect visually)")
        plt.tight_layout(); savefig("duplicate_candidates"); plt.show()

---
## Step 3 · Exploratory data analysis

In [ ]:
all_df = pd.concat(splits.values(), ignore_index=True)
by_type = pd.crosstab(all_df["tumor_type"], all_df["split"])[SPLIT_NAMES]
by_plane = pd.crosstab(all_df["plane"], all_df["split"])[SPLIT_NAMES]
display(by_type)
display(by_plane)

fig, axes = plt.subplots(1, 3, figsize=(17, 4.2))
by_type.plot.bar(ax=axes[0], rot=0, title="Images per class and split")
by_plane.plot.bar(ax=axes[1], rot=0, title="Images per plane and split")
ct = pd.crosstab(tumour_df["tumor_type"], tumour_df["plane"])
im = axes[2].imshow(ct.values, cmap="Blues")
axes[2].set_xticks(range(ct.shape[1]), ct.columns); axes[2].set_yticks(range(ct.shape[0]), ct.index)
for (i, j), v in np.ndenumerate(ct.values):
    axes[2].text(j, i, v, ha="center", va="center")
axes[2].set_title("Tumour type × plane (segmentation set)"); axes[2].grid(False)
for ax in axes[:2]:
    ax.set_xlabel(""); ax.legend(fontsize=7)
plt.tight_layout(); savefig("eda_distribution"); plt.show()

### 3.1 Tumour size and location

Tumour area as a fraction of the slice, and a **spatial prior** (average mask) per tumour type.

In [ ]:
MAX_IMAGES = 2000
size_df = tumour_df.sample(n=min(MAX_IMAGES, len(tumour_df)), random_state=SEED).copy()
masks_small = {}
fractions = []
for _, r in size_df.iterrows():
    m = read_mask(DATA_ROOT / r["mask"])
    fractions.append(m.mean())
    masks_small.setdefault(r["tumor_type"], []).append(cv2.resize(m.astype(np.float32), (128, 128)))
size_df["tumor_pct"] = np.array(fractions) * 100

display(size_df.groupby("tumor_type")["tumor_pct"].describe(percentiles=[.1, .5, .9]).round(2))

types = sorted(masks_small)
fig, axes = plt.subplots(1, 2 + len(types), figsize=(5 * 2 + 3 * len(types), 3.6),
                         gridspec_kw={"width_ratios": [1.5, 1.5] + [1] * len(types)})
for t in types:
    axes[0].hist(size_df.loc[size_df["tumor_type"] == t, "tumor_pct"], bins=40, alpha=0.5, label=t)
axes[0].set_xlabel("tumour area (% of slice)"); axes[0].set_ylabel("images"); axes[0].legend()
axes[0].set_title("Tumour size distribution")
axes[1].boxplot([size_df.loc[size_df["tumor_type"] == t, "tumor_pct"] for t in types])
axes[1].set_xticks(range(1, len(types) + 1), types); axes[1].set_yscale("log")
axes[1].set_ylabel("% of slice (log)"); axes[1].set_title("Tumour size by type")
for ax, t in zip(axes[2:], types):
    ax.imshow(np.mean(masks_small[t], axis=0), cmap="inferno"); ax.set_title(f"location prior: {t}")
    ax.axis("off")
plt.tight_layout(); savefig("eda_size_location"); plt.show()
print(f"Median tumour size: {size_df['tumor_pct'].median():.2f}% of the slice → "
      "predicting 'background everywhere' would already give > 98% pixel accuracy, hence Dice/IoU.")

### 3.2 Example slices with expert masks

In [ ]:
from src.analysis import fill_overlay, contour_overlay, error_overlay

fig, axes = plt.subplots(4, 4, figsize=(12, 12))
groups = [("glioma", False), ("meningioma", False), ("pituitary", False), ("no_tumor", True)]
for i, (t, healthy) in enumerate(groups):
    src_df = splits["healthy_train"] if healthy else splits["train"][splits["train"]["tumor_type"] == t]
    for j, (_, r) in enumerate(src_df.sample(n=4, random_state=i).iterrows()):
        image, mask = load_pair(DATA_ROOT, r)
        axes[i, j].imshow(contour_overlay(fill_overlay(image, mask, (255, 200, 0), 0.3), mask))
        axes[i, j].set_title(f"{t} · {r['plane']}", fontsize=9); axes[i, j].axis("off")
plt.tight_layout(); savefig("eda_samples"); plt.show()

---
## Step 4 · Pipeline sanity checks

Before spending hours on training, each building block is verified:
data loader output, augmentation, model forward passes, loss and metric behaviour.

**Preprocessing:** grayscale → 3 channels (so ImageNet encoders can be used), resize 512 → 256,
ImageNet normalisation, mask binarisation.
**Augmentation (training only):** horizontal flip, affine (±15°, ±10 % scale, ±5 % shift),
brightness/contrast, gamma, mild elastic deformation.

In [ ]:
from torch.utils.data import DataLoader
from src.dataset import BriscSegDataset
from src.transforms import get_train_transforms, get_eval_transforms, get_preview_transforms, IMAGENET_MEAN, IMAGENET_STD

loader = DataLoader(BriscSegDataset(splits["train"], DATA_ROOT, get_train_transforms(256)), batch_size=8, shuffle=True)
images, masks = next(iter(loader))
print("image batch:", tuple(images.shape), images.dtype, "| mask batch:", tuple(masks.shape), masks.dtype)
check("batch shapes are [B,3,256,256] and [B,1,256,256]", images.shape[1:] == (3, 256, 256) and masks.shape[1:] == (1, 256, 256))
check("mask values are binary", set(torch.unique(masks).tolist()) <= {0.0, 1.0}, str(torch.unique(masks).tolist()))
print(f"normalised image mean {images.mean():.3f}, std {images.std():.3f} (≈ 0 / 1 expected)")

healthy_ds = BriscSegDataset(splits["healthy_train"], DATA_ROOT, get_eval_transforms(256))
check("healthy slice → all-zero mask", healthy_ds[0][1].sum().item() == 0)

# augmentation preview
aug = get_preview_transforms(256)
r = splits["train"].iloc[0]
image, mask = load_pair(DATA_ROOT, r)
fig, axes = plt.subplots(1, 6, figsize=(18, 3.2))
axes[0].imshow(fill_overlay(cv2.resize(image, (256, 256)), cv2.resize(mask, (256, 256), interpolation=cv2.INTER_NEAREST)))
axes[0].set_title("original (resized)")
for ax in axes[1:]:
    out = aug(image=image, mask=mask)
    ax.imshow(fill_overlay(out["image"], out["mask"])); ax.set_title("augmented")
for ax in axes:
    ax.axis("off")
plt.tight_layout(); savefig("augmentations"); plt.show()

In [ ]:
from src.models import build_model, count_parameters
from src.losses import BCEDiceLoss
from src.metrics import confusion_counts, summarize, validation_summary

if RUN_MODEL_CHECKS:
    rows = []
    for run_name, cfg_path in VARIANTS.items():
        cfg = load_config(cfg_path)
        model = build_model(dict(cfg["model"], encoder_weights=None)).to(DEVICE).eval()
        with torch.no_grad():
            x = torch.zeros(2, 3, 256, 256, device=DEVICE)
            t0 = time.perf_counter(); y = model(x)
            if DEVICE.type == "mps": torch.mps.synchronize()
        rows.append({"run": run_name, "architecture": cfg["model"]["name"],
                     "encoder": cfg["model"].get("encoder_name") if cfg["model"]["name"] != "unet_scratch" else "—",
                     "params (M)": count_parameters(model) / 1e6, "output": tuple(y.shape),
                     "healthy slices": cfg["data"]["include_healthy"], "lr": cfg["train"]["lr"],
                     "encoder lr ×": cfg["train"]["encoder_lr_mult"], "max epochs": cfg["train"]["epochs"]})
        check(f"{run_name}: forward pass gives [B,1,256,256]", tuple(y.shape) == (2, 1, 256, 256))
        del model
    if DEVICE.type == "mps":
        torch.mps.empty_cache()
    display(pd.DataFrame(rows))

# loss & metric behaviour on hand-made cases
target = torch.zeros(1, 1, 64, 64); target[..., 20:40, 20:40] = 1
perfect, inverted = (target * 2 - 1) * 20, (1 - target * 2) * 20
loss_fn = BCEDiceLoss()
check("loss ≈ 0 for a perfect prediction", loss_fn(perfect, target).item() < 1e-3, f"{loss_fn(perfect, target).item():.5f}")
check("loss is large for an inverted prediction", loss_fn(inverted, target).item() > 5, f"{loss_fn(inverted, target).item():.2f}")
gt = target[0].numpy().astype(bool)
half = gt.copy(); half[:, :, 30:] = False
check("Dice = 1 for identical masks", summarize(confusion_counts(gt, gt))["dice"] == 1.0)
check("Dice = 2/3 when half of the tumour is found", np.isclose(summarize(confusion_counts(half, gt))["dice"], 2 / 3))
empty = np.zeros_like(gt)
vs = validation_summary(np.concatenate([confusion_counts(empty, empty), confusion_counts(gt, empty)]))
check("healthy slice: empty prediction counts as correct, any blob as false positive", vs["healthy_fp_rate"] == 0.5)

---
## Step 5 · Smoke test on synthetic data (optional)

Runs the complete train loop for 2 epochs on a tiny synthetic dataset with the same folder layout as BRISC,
to catch errors in minutes rather than after an hour.

In [ ]:
if RUN_SMOKE_TEST:
    run(["scripts/make_synthetic_data.py"])
    run(["-m", "src.prepare_data", "--data-root", "data/synthetic", "--out", "data/splits_synthetic"])
    run(["-m", "src.train", "--config", "configs/unet_scratch.yaml", "--smoke",
         "--data-root", "data/synthetic", "--splits-dir", "data/splits_synthetic"])
    display(pd.read_csv(RUNS / "unet_scratch_smoke" / "history.csv"))
else:
    print("Smoke test skipped (RUN_SMOKE_TEST = False).")

---
## Step 6 · Training

**Setup (all variants):** BCE + Dice loss, AdamW (weight decay 1e-4), 1 warm-up epoch then cosine decay,
gradient clipping 1.0, batch 8 at 256 px, early stopping on validation Dice, best checkpoint kept.
Pretrained encoders use a 0.3× learning rate so the ImageNet features are adapted gently.

Equivalent Terminal command (recommended; add `--resume` to continue after an interruption):

```bash
python -m src.train --config configs/unet_resnet34.yaml
```

The submitted run trained for 40 epochs (≈ 116 min on the M4); the best validation Dice (0.883) was reached at epoch 35.

In [ ]:
def is_trained(r):
    return (RUNS / r / "summary.json").exists() and (RUNS / r / "best.pt").exists()

for run_name, cfg_path in VARIANTS.items():
    if is_trained(run_name) and SKIP_COMPLETED:
        print(f"✔ {run_name}: already trained")
        continue
    if not RUN_TRAINING:
        print(f"✘ {run_name}: not trained yet → run `python -m src.train --config {cfg_path}` "
              "in Terminal, or set RUN_TRAINING = True")
        continue
    resume = (RUNS / run_name / "last.pt").exists()
    run(["-m", "src.train", "--config", cfg_path] + (["--resume"] if resume else []))

def load_run(r):
    d = RUNS / r
    return {"summary": json.loads((d / "summary.json").read_text()),
            "history": pd.read_csv(d / "history.csv"),
            "config": yaml.safe_load((d / "config.yaml").read_text())}

TRAINED = {r: load_run(r) for r in VARIANTS if is_trained(r)}
print("\nTrained runs available for analysis:", list(TRAINED))

### 6.1 Training summary and consistency checks

In [ ]:
def tumour_dice_col(h):
    return "val_dice_tumor" if "val_dice_tumor" in h and h["val_dice_tumor"].notna().any() else "val_dice"

rows = []
for r, d in TRAINED.items():
    h, s = d["history"], d["summary"]
    best_row = h.loc[h["val_dice"].idxmax()]
    rows.append({
        "model": label(r), "params (M)": s["parameters"] / 1e6, "epochs run": len(h),
        "best epoch": s["best_epoch"], "best val Dice (selection)": s["best_val_dice"],
        "val Dice tumour @best": best_row[tumour_dice_col(h)],
        "stopped early": len(h) < d["config"]["train"]["epochs"],
        "mean epoch (s)": h["epoch_time_s"].mean(), "total train (min)": h["epoch_time_s"].sum() / 60,
    })
    check(f"{r}: best epoch in history matches checkpoint", int(best_row["epoch"]) == int(s["best_epoch"]),
          f"history {int(best_row['epoch'])} vs summary {s['best_epoch']}")
    check(f"{r}: best val Dice matches history", np.isclose(best_row["val_dice"], s["best_val_dice"]))
TRAIN_TABLE = pd.DataFrame(rows)
TRAIN_TABLE

### 6.2 Learning curves

What to look for: **(a)** training vs validation loss (a widening gap signals over-fitting),
**(b)** validation Dice plateau (when more epochs stop paying off),
**(c)** the learning-rate schedule, **(d)** epoch duration (thermal throttling / background load on a fanless laptop).

In [ ]:
n = len(TRAINED)
if n:
    fig, axes = plt.subplots(1, n, figsize=(4.6 * n, 3.6), squeeze=False, sharey=True)
    for ax, (r, d) in zip(axes[0], TRAINED.items()):
        h = d["history"]
        ax.plot(h["epoch"], h["train_loss"], label="train")
        ax.plot(h["epoch"], h["val_loss"], label="validation")
        ax.axvline(d["summary"]["best_epoch"], color="grey", ls=":", label="best epoch")
        ax.set_title(label(r)); ax.set_xlabel("epoch"); ax.set_yscale("log")
    axes[0, 0].set_ylabel("BCE + Dice loss (log)"); axes[0, 0].legend()
    plt.tight_layout(); savefig("loss_curves"); plt.show()

    fig, axes = plt.subplots(2, 2, figsize=(14, 8))
    for r, d in TRAINED.items():
        h, c = d["history"], COLORS[r]
        axes[0, 0].plot(h["epoch"], h[tumour_dice_col(h)], color=c, label=label(r))
        axes[0, 1].plot(h["epoch"], h["val_loss"] - h["train_loss"], color=c, label=label(r))
        axes[1, 0].plot(h["epoch"], h["lr"], color=c, label=label(r))
        axes[1, 1].plot(h["epoch"], h["epoch_time_s"], color=c, marker=".", label=label(r))
    axes[0, 0].set_title("(b) Validation Dice — tumour slices"); axes[0, 0].set_ylim(bottom=0.5)
    axes[0, 1].set_title("(a) Generalisation gap: val loss − train loss"); axes[0, 1].axhline(0, color="k", lw=0.8)
    axes[1, 0].set_title("(c) Learning rate (decoder)"); axes[1, 0].set_yscale("log")
    axes[1, 1].set_title("(d) Epoch duration (s)")
    for ax in axes.flat:
        ax.set_xlabel("epoch"); ax.legend(fontsize=8)
    plt.tight_layout(); savefig("training_dynamics"); plt.show()

if HEALTHY_RUN in TRAINED:
    h = TRAINED[HEALTHY_RUN]["history"]
    fig, ax1 = plt.subplots(figsize=(8, 3.6))
    ax1.plot(h["epoch"], h["val_dice_tumor"], color="tab:blue", label="val Dice (tumour slices)")
    ax1.plot(h["epoch"], h["val_dice"], color="tab:green", label="val Dice (tumour + healthy)")
    ax2 = ax1.twinx()
    ax2.plot(h["epoch"], 100 * h["val_healthy_fp_rate"], color="tab:red", ls="--", label="healthy FP rate (%)")
    ax1.set_xlabel("epoch"); ax1.set_ylabel("Dice"); ax2.set_ylabel("healthy slices with any FP (%)")
    ax1.legend(loc="lower left"); ax2.legend(loc="upper right"); ax2.grid(False)
    ax1.set_title(f"{label(HEALTHY_RUN)}: segmentation quality vs false positives during training")
    plt.tight_layout(); savefig("healthy_run_training"); plt.show()

---
## Step 7 · Test-set evaluation

`src.evaluate` predicts at 256 px, upsamples the probability map to the native 512 px and scores it against
the original expert masks (no down-sampling bias). It also scores the 140 healthy test slices,
measures GPU inference time and saves per-image results and galleries.

In [ ]:
def is_evaluated(r, name="eval"):
    return (RUNS / r / name / "metrics.json").exists()

for r in TRAINED:
    if is_evaluated(r) and SKIP_COMPLETED:
        print(f"✔ {r}: already evaluated")
    elif RUN_EVALUATION:
        run(["-m", "src.evaluate", "--run", f"{RUNS.relative_to(ROOT)}/{r}"])
    else:
        print(f"✘ {r}: not evaluated (RUN_EVALUATION = False)")

def load_eval(r, name="eval"):
    d = RUNS / r / name
    healthy_path = d / "healthy_per_image.csv"
    return {"metrics": json.loads((d / "metrics.json").read_text()),
            "test": pd.read_csv(d / "test_per_image.csv"),
            "healthy": pd.read_csv(healthy_path) if healthy_path.exists() else pd.DataFrame()}

EVAL = {r: load_eval(r) for r in TRAINED if is_evaluated(r)}
print("Evaluated runs:", list(EVAL))

### 7.1 Result validation

The headline numbers in `metrics.json` are recomputed independently from the per-image CSV files,
and basic invariants are checked (sample counts, value ranges, pixel bookkeeping).

In [ ]:
from src.analysis import overall_metrics, healthy_metrics, grouped_metrics

for r, e in EVAL.items():
    t, hl, m = e["test"], e["healthy"], e["metrics"]
    recomputed = overall_metrics(t)
    keys = ["dice", "iou", "precision", "recall", "global_dice"]
    check(f"{r}: metrics.json reproduced from per-image CSV",
          all(np.isclose(recomputed[k], m["overall"][k]) for k in keys))
    check(f"{r}: all 860 test slices scored", len(t) == 860 and t["stem"].is_unique, f"{len(t)} rows")
    check(f"{r}: all 140 healthy slices scored", len(hl) == 140, f"{len(hl)} rows")
    check(f"{r}: scores within [0, 1]", t[["dice", "iou", "precision", "recall"]].stack().between(0, 1).all())
    check(f"{r}: TP+FP+FN+TN equals the pixel count",
          (t[["tp", "fp", "fn", "tn"]].sum(axis=1) == t["height"] * t["width"]).all())
    check(f"{r}: every test slice has a tumour in the ground truth", (t["gt_area_frac"] > 0).all())
    check(f"{r}: IoU ≤ Dice for every slice", (t["iou"] <= t["dice"] + 1e-9).all())
    if len(hl):
        rec_h = healthy_metrics(hl)
        check(f"{r}: healthy FP rate reproduced", np.isclose(rec_h["fp_rate_largest_ge_50px"], m["healthy"]["fp_rate_largest_ge_50px"]))

    val_dice = TRAINED[r]["history"].pipe(lambda h: h.loc[h["val_dice"].idxmax(), tumour_dice_col(h)])
    gap = val_dice - m["overall"]["dice"]
    check(f"{r}: validation → test Dice drop below 0.05 (no selection over-fit)", gap < 0.05,
          f"val {val_dice:.3f} vs test {m['overall']['dice']:.3f}")

### 7.2 Model comparison

In [ ]:
if EVAL:
    run(["-m", "src.compare_runs", "--runs", *[f"{RUNS.relative_to(ROOT)}/{r}" for r in EVAL]])

rows = []
for r, e in EVAL.items():
    m, t = e["metrics"], e["test"]
    rows.append({
        "model": label(r),
        "params (M)": TRAINED[r]["summary"]["parameters"] / 1e6,
        "Dice mean": m["overall"]["dice"], "Dice median": m["overall"]["dice_median"],
        "IoU": m["overall"]["iou"], "precision": m["overall"]["precision"], "recall": m["overall"]["recall"],
        "global Dice": m["overall"]["global_dice"],
        "slices Dice < 0.5": int((t["dice"] < 0.5).sum()),
        "healthy FP ≥ 50 px": m["healthy"].get("fp_rate_largest_ge_50px", np.nan),
        "ms / image": m["speed"]["ms_per_image_forward"],
    })
RESULTS = pd.DataFrame(rows).set_index("model") if rows else pd.DataFrame()
BEST_RUN = max(EVAL, key=lambda r: EVAL[r]["metrics"]["overall"]["dice"]) if EVAL else None
print("Best test Dice:", label(BEST_RUN) if BEST_RUN else "—")
display(RESULTS.style.format("{:.3f}")
        .format("{:.1f}", subset=["params (M)", "ms / image"])
        .format("{:d}", subset=["slices Dice < 0.5"])
        .format("{:.1%}", subset=["healthy FP ≥ 50 px"])
        .highlight_max(subset=["Dice mean", "Dice median", "IoU", "global Dice"], color="#c6efce")
        .highlight_min(subset=["healthy FP ≥ 50 px", "ms / image", "slices Dice < 0.5"], color="#c6efce")
        if len(RESULTS) else "No evaluated runs yet.")

In [ ]:
if EVAL:
    fig, axes = plt.subplots(1, 3, figsize=(18, 4.2))
    for ax, key, title in zip(axes, ["by_tumor_type", "by_plane", "by_tumor_size"],
                              ["tumour type", "imaging plane", "tumour size (tertiles)"]):
        groups = list(next(iter(EVAL.values()))["metrics"][key])
        x, w = np.arange(len(groups)), 0.8 / len(EVAL)
        for i, (r, e) in enumerate(EVAL.items()):
            vals = [e["metrics"][key][g]["dice"] for g in groups]
            bars = ax.bar(x + (i - (len(EVAL) - 1) / 2) * w, vals, w, label=label(r), color=COLORS[r])
            ax.bar_label(bars, fmt="%.2f", fontsize=6, padding=1)
        ax.set_xticks(x, groups); ax.set_ylim(0.5, 1.0); ax.set_title(f"Mean test Dice by {title}")
    axes[0].legend(fontsize=8, loc="lower left")
    plt.tight_layout(); savefig("dice_breakdowns"); plt.show()

### 7.3 Distribution of per-slice Dice

Means hide the shape of the error distribution. The **ECDF** shows which share of slices falls below a Dice value
(curves further right/lower are better); the violin plot shows the long tail of hard cases.

In [ ]:
if EVAL:
    fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))
    for r, e in EVAL.items():
        d = np.sort(e["test"]["dice"].values)
        axes[0].step(d, np.arange(1, len(d) + 1) / len(d), where="post", label=label(r), color=COLORS[r])
    for thr in (0.5, 0.8):
        axes[0].axvline(thr, color="grey", ls=":", lw=1)
    axes[0].set_xlabel("Dice"); axes[0].set_ylabel("share of test slices ≤ Dice")
    axes[0].set_title("ECDF of per-slice Dice"); axes[0].legend(fontsize=8)

    parts = axes[1].violinplot([e["test"]["dice"] for e in EVAL.values()], showmedians=True, showextrema=False)
    for body, r in zip(parts["bodies"], EVAL):
        body.set_facecolor(COLORS[r]); body.set_alpha(0.6)
    axes[1].set_xticks(range(1, len(EVAL) + 1), [label(r) for r in EVAL], rotation=12)
    axes[1].set_ylabel("Dice"); axes[1].set_title("Per-slice Dice distribution (line = median)")
    plt.tight_layout(); savefig("dice_distribution"); plt.show()

    display(pd.DataFrame({label(r): {f"Dice < {t}": f"{(e['test']['dice'] < t).mean():.1%}" for t in (0.1, 0.5, 0.8, 0.9)}
                          for r, e in EVAL.items()}))

### 7.4 Error structure: tumour size, type × plane, over- vs under-segmentation

In [ ]:
if EVAL:
    fig, axes = plt.subplots(1, 2, figsize=(15, 4.8))
    bins = np.logspace(np.log10(EVAL[BEST_RUN]["test"]["gt_area_frac"].min() * 100),
                       np.log10(EVAL[BEST_RUN]["test"]["gt_area_frac"].max() * 100), 12)
    for r, e in EVAL.items():
        t = e["test"].assign(pct=lambda d: d["gt_area_frac"] * 100)
        if r == BEST_RUN:
            axes[0].scatter(t["pct"], t["dice"], s=6, alpha=0.3, color=COLORS[r])
        med = t.groupby(pd.cut(t["pct"], bins), observed=True)["dice"].median()
        centers = [iv.mid for iv in med.index]
        axes[0].plot(centers, med.values, marker="o", color=COLORS[r], label=f"{label(r)} (binned median)")
    axes[0].set_xscale("log"); axes[0].set_xlabel("tumour area (% of slice, log)"); axes[0].set_ylabel("Dice")
    axes[0].set_title(f"Dice vs tumour size (dots: {label(BEST_RUN)})"); axes[0].legend(fontsize=7)

    t = EVAL[BEST_RUN]["test"]
    sc = axes[1].scatter(t["recall"], t["precision"], c=t["tumor_type"].astype("category").cat.codes,
                         cmap="viridis", s=8, alpha=0.5)
    axes[1].plot([0, 1], [0, 1], color="grey", ls=":")
    axes[1].set_xlabel("recall (share of tumour found)"); axes[1].set_ylabel("precision (share of prediction that is tumour)")
    axes[1].set_title(f"{label(BEST_RUN)}: over- vs under-segmentation per slice")
    handles = [plt.Line2D([], [], marker="o", ls="", color=sc.cmap(sc.norm(i)), label=c)
               for i, c in enumerate(t["tumor_type"].astype("category").cat.categories)]
    axes[1].legend(handles=handles, fontsize=8)
    axes[1].text(0.02, 0.02, "below diagonal: over-segmentation\nabove diagonal: under-segmentation",
                 transform=axes[1].transAxes, fontsize=8)
    plt.tight_layout(); savefig("error_structure"); plt.show()

    fig, axes = plt.subplots(1, len(EVAL), figsize=(4.2 * len(EVAL), 3.4), squeeze=False)
    for ax, (r, e) in zip(axes[0], EVAL.items()):
        pv = e["test"].pivot_table(index="tumor_type", columns="plane", values="dice", aggfunc="mean")
        ax.imshow(pv.values, cmap="RdYlGn", vmin=0.6, vmax=1.0)
        ax.set_xticks(range(pv.shape[1]), pv.columns); ax.set_yticks(range(pv.shape[0]), pv.index)
        for (i, j), v in np.ndenumerate(pv.values):
            ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=9)
        ax.set_title(label(r), fontsize=10); ax.grid(False)
    plt.suptitle("Mean test Dice: tumour type × plane"); plt.tight_layout(); savefig("type_plane_heatmap"); plt.show()

    fails = pd.DataFrame({label(r): e["test"].loc[e["test"]["dice"] < 0.5, "tumor_type"].value_counts()
                          for r, e in EVAL.items()}).fillna(0).astype(int)
    print("Failure cases (Dice < 0.5) per tumour type:")
    display(fails)

---
## Step 8 · Statistical analysis

* **Bootstrap 95 % confidence intervals** (2,000 resamples of the 860 test slices) for the mean Dice.
* **Per-group confidence intervals** (tumour type, plane, size) — are the differences between groups real?
* **Mann–Whitney U tests** between tumour types (independent groups of slices).
* **Paired Wilcoxon signed-rank test** against the baseline when more than one model is available
  (all models are scored on the *same* slices). Inference-time variants are compared the same way in Step 10.

In [ ]:
from scipy.stats import wilcoxon

def bootstrap_ci(values, n_boot=2000, seed=SEED):
    rng = np.random.default_rng(seed)
    values = np.asarray(values)
    means = values[rng.integers(0, len(values), (n_boot, len(values)))].mean(axis=1)
    return np.percentile(means, [2.5, 97.5])

rows = []
base = EVAL.get(BASELINE_RUN)
for r, e in EVAL.items():
    lo, hi = bootstrap_ci(e["test"]["dice"])
    row = {"model": label(r), "mean Dice": e["test"]["dice"].mean(), "95% CI low": lo, "95% CI high": hi}
    if base is not None and r != BASELINE_RUN:
        paired = e["test"][["stem", "dice"]].merge(base["test"][["stem", "dice"]], on="stem", suffixes=("", "_base"))
        diff = paired["dice"] - paired["dice_base"]
        row.update({"Δ vs baseline": diff.mean(), "slices better": int((diff > 0).sum()),
                    "slices worse": int((diff < 0).sum())})
        try:
            row["Wilcoxon p"] = wilcoxon(paired["dice"], paired["dice_base"]).pvalue
        except ValueError:  # identical results
            row["Wilcoxon p"] = 1.0
    rows.append(row)
STATS = pd.DataFrame(rows).set_index("model") if rows else pd.DataFrame()
display(STATS)

if len(STATS):
    fig, ax = plt.subplots(figsize=(8, 0.6 * len(STATS) + 1.5))
    y = np.arange(len(STATS))
    ax.errorbar(STATS["mean Dice"], y, xerr=[STATS["mean Dice"] - STATS["95% CI low"], STATS["95% CI high"] - STATS["mean Dice"]],
                fmt="o", capsize=4)
    ax.set_yticks(y, STATS.index); ax.set_xlabel("mean test Dice (95% bootstrap CI)")
    ax.set_title("Mean test Dice with uncertainty")
    plt.tight_layout(); savefig("dice_ci"); plt.show()
if BEST_RUN:
    from scipy.stats import mannwhitneyu
    t = EVAL[BEST_RUN]["test"]
    group_rows = []
    for col in ["tumor_type", "plane", "size_bucket"]:
        for g, sub in t.groupby(col, observed=True):
            lo, hi = bootstrap_ci(sub["dice"])
            group_rows.append({"grouping": col, "group": g, "n": len(sub), "mean Dice": sub["dice"].mean(),
                               "95% CI low": lo, "95% CI high": hi})
    GROUP_CI = pd.DataFrame(group_rows)
    display(GROUP_CI.set_index(["grouping", "group"]))

    fig, ax = plt.subplots(figsize=(9, 4))
    ylab = [f"{r.grouping.replace('_', ' ')}: {r.group}" for r in GROUP_CI.itertuples()]
    y = np.arange(len(GROUP_CI))
    ax.errorbar(GROUP_CI["mean Dice"], y, fmt="o", capsize=4,
                xerr=[GROUP_CI["mean Dice"] - GROUP_CI["95% CI low"], GROUP_CI["95% CI high"] - GROUP_CI["mean Dice"]])
    ax.set_yticks(y, ylab); ax.invert_yaxis(); ax.set_xlabel("mean test Dice (95% bootstrap CI)")
    ax.set_title(f"{label(BEST_RUN)}: Dice by group with uncertainty")
    plt.tight_layout(); savefig("group_ci"); plt.show()

    pairs = []
    for a, b in itertools.combinations(sorted(t["tumor_type"].unique()), 2):
        p = mannwhitneyu(t.loc[t["tumor_type"] == a, "dice"], t.loc[t["tumor_type"] == b, "dice"]).pvalue
        pairs.append({"comparison": f"{a} vs {b}", "Mann–Whitney p": p, "significant (p < 0.05)": p < 0.05})
    display(pd.DataFrame(pairs))
print("p < 0.05 → the difference is unlikely to be chance (on this test set).")

---
## Step 9 · Healthy slices and image-level tumour detection

The segmentation data contains no healthy slices, so a model trained only on it may learn that *every*
slice contains a tumour. In a real application most scans are healthy, so this matters.
The baseline evaluation showed exactly this problem (predicted blobs ≥ 50 px on ≈ 67 % of healthy slices).

Two views:
1. **False-positive rate** on the 140 healthy test slices at several blob-size thresholds.
2. **Image-level detection**: using the largest predicted blob as a "tumour present" score,
   how well does each model separate the 860 tumour slices from the 140 healthy ones? (ROC / AUC)

In [ ]:
from sklearn.metrics import roc_auc_score, roc_curve

HEALTHY_EVAL = {r: e for r, e in EVAL.items() if len(e["healthy"])}
if HEALTHY_EVAL:
    fp_table = pd.DataFrame({label(r): {f"blob ≥ {t} px": (e["healthy"]["largest_component_px"] >= t).mean()
                                         for t in (1, 50, 200, 1000)}
                             for r, e in HEALTHY_EVAL.items()}).T
    display(fp_table.style.format("{:.1%}"))

    fig, axes = plt.subplots(1, 3, figsize=(19, 4.8))
    fp_table.plot.bar(ax=axes[0], rot=12)
    axes[0].set_ylabel("share of healthy slices"); axes[0].set_title("False positives on healthy test slices")

    det_rows = []
    for r, e in HEALTHY_EVAL.items():
        y = np.r_[np.ones(len(e["test"])), np.zeros(len(e["healthy"]))]
        for ax, col, name in [(axes[1], "largest_component_px", "largest blob"), (axes[2], "max_prob", "max probability")]:
            score = np.r_[e["test"][col], e["healthy"][col]]
            fpr, tpr, _ = roc_curve(y, score)
            auc = roc_auc_score(y, score)
            ax.plot(fpr, tpr, color=COLORS[r], label=f"{label(r)} (AUC {auc:.3f})")
            det_rows.append({"model": label(r), "score": name, "AUC": auc})
        tumour_hit = (e["test"]["largest_component_px"] >= 50).mean()
        healthy_clean = (e["healthy"]["largest_component_px"] < 50).mean()
        det_rows.append({"model": label(r), "score": "blob ≥ 50 px rule", "sensitivity": tumour_hit,
                         "specificity": healthy_clean, "balanced accuracy": (tumour_hit + healthy_clean) / 2})
    for ax, name in [(axes[1], "largest predicted blob"), (axes[2], "maximum pixel probability")]:
        ax.plot([0, 1], [0, 1], color="grey", ls=":")
        ax.set_xlabel("false-positive rate (healthy slices)"); ax.set_ylabel("true-positive rate (tumour slices)")
        ax.set_title(f"Tumour-present ROC — score: {name}"); ax.legend(fontsize=8)
    plt.tight_layout(); savefig("healthy_detection"); plt.show()

    DETECTION = pd.DataFrame(det_rows)
    display(DETECTION.pivot_table(index="model", columns="score", values="AUC"))
    display(DETECTION.dropna(subset=["sensitivity"])[["model", "sensitivity", "specificity", "balanced accuracy"]]
            .set_index("model").style.format("{:.1%}"))

    fig, ax = plt.subplots(figsize=(9, 3.8))
    for r, e in HEALTHY_EVAL.items():
        ax.hist(np.log10(e["healthy"]["largest_component_px"] + 1), bins=30, alpha=0.5, color=COLORS[r], label=label(r))
    ax.set_xlabel("log10(largest predicted blob on healthy slice + 1 px)"); ax.set_ylabel("healthy slices")
    ax.set_title("Size of false-positive blobs (0 = clean prediction)"); ax.legend(fontsize=8)
    plt.tight_layout(); savefig("healthy_fp_sizes"); plt.show()

*Interpretation guide:* an AUC close to 0.5 means the model's output barely differs between healthy and tumour slices
(it "always finds something"); the "blob ≥ 50 px" rule shows the sensitivity/specificity of using the segmentation
directly as a tumour detector. The model was trained on tumour slices only, so low specificity is expected — the
proposed fix (training with healthy slices, `configs/unet_resnet34_healthy.yaml`) is discussed in the README.

---
## Step 10 · Improvement experiments

### 10.1 Post-processing and test-time augmentation (TTA)

* `--min-area 200` removes predicted blobs smaller than 200 px (native resolution).
* `--tta` averages the prediction with that of the horizontally flipped slice (2× inference cost).

In [ ]:
POSTPROC = {"eval": [], "eval_min200": ["--min-area", "200"], "eval_tta": ["--tta"],
            "eval_tta_min200": ["--tta", "--min-area", "200"]}
POSTPROC_RUNS = [r for r in dict.fromkeys([BASELINE_RUN, BEST_RUN, HEALTHY_RUN]) if r in EVAL]

for r in POSTPROC_RUNS:
    for name, extra in POSTPROC.items():
        if is_evaluated(r, name) and SKIP_COMPLETED:
            continue
        if RUN_POSTPROC_EXPERIMENTS:
            run(["-m", "src.evaluate", "--run", f"{RUNS.relative_to(ROOT)}/{r}", *extra])

rows = []
for r in POSTPROC_RUNS:
    for name in POSTPROC:
        if not is_evaluated(r, name):
            continue
        m = load_eval(r, name)["metrics"]
        rows.append({"model": label(r), "variant": name, "Dice mean": m["overall"]["dice"],
                     "Dice median": m["overall"]["dice_median"], "precision": m["overall"]["precision"],
                     "recall": m["overall"]["recall"],
                     "healthy FP ≥ 1 px": m["healthy"].get("fp_rate_largest_ge_1px", np.nan),
                     "healthy FP ≥ 50 px": m["healthy"].get("fp_rate_largest_ge_50px", np.nan),
                     "ms / image": m["speed"]["ms_per_image_forward"]})
POSTPROC_TABLE = pd.DataFrame(rows)
if len(POSTPROC_TABLE):
    display(POSTPROC_TABLE.set_index(["model", "variant"]).style
            .format("{:.3f}").format("{:.1%}", subset=["healthy FP ≥ 1 px", "healthy FP ≥ 50 px"])
            .format("{:.1f}", subset=["ms / image"]))
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    for ax, col in zip(axes, ["Dice mean", "healthy FP ≥ 50 px"]):
        POSTPROC_TABLE.pivot(index="model", columns="variant", values=col).reindex(columns=list(POSTPROC)).plot.bar(ax=ax, rot=0)
        ax.set_title(col); ax.legend(fontsize=8)
    axes[0].set_ylim(0.6, 1.0)
    plt.tight_layout(); savefig("postprocessing"); plt.show()

    from scipy.stats import wilcoxon
    paired_rows = []
    for r in POSTPROC_RUNS:
        base_t = load_eval(r, "eval")["test"][["stem", "dice"]]
        for name in list(POSTPROC)[1:]:
            if not is_evaluated(r, name):
                continue
            other = load_eval(r, name)["test"][["stem", "dice"]]
            m = other.merge(base_t, on="stem", suffixes=("", "_base"))
            diff = m["dice"] - m["dice_base"]
            try:
                p = wilcoxon(m["dice"], m["dice_base"]).pvalue
            except ValueError:
                p = 1.0
            paired_rows.append({"model": label(r), "variant": name, "mean ΔDice": diff.mean(),
                                "slices better": int((diff > 1e-9).sum()), "slices worse": int((diff < -1e-9).sum()),
                                "Wilcoxon p": p})
    if paired_rows:
        print("Paired comparison with the baseline evaluation (same slices):")
        display(pd.DataFrame(paired_rows))

### 10.2 Decision threshold tuned on the validation set

The default threshold is 0.5. Here the threshold is chosen on the **validation** set
(tumour + held-out healthy validation slices) — never on the test set — by maximising Dice over both
(an empty prediction on a healthy slice scores 1, any blob scores 0). The test set is then re-evaluated once
at that threshold. Validation metrics are computed at native resolution, like the test evaluation.

In [ ]:
from src.data_index import load_train_val_frames

THRESHOLDS = np.round(np.arange(0.10, 0.951, 0.05), 2)
_, VAL_DF = load_train_val_frames(SPLITS, include_healthy=True, healthy_val_fraction=0.15, seed=SEED)
print(f"Validation set for the sweep: {len(VAL_DF)} slices "
      f"({(VAL_DF['mask'] != '').sum()} tumour, {(VAL_DF['mask'] == '').sum()} healthy)")

sweeps = []
for r in TRAINED:
    cache = RUNS / r / "threshold_sweep_val.csv"
    if cache.exists() and SKIP_COMPLETED:
        sweeps.append(pd.read_csv(cache)); continue
    if not RUN_THRESHOLD_SWEEP:
        continue
    from src.evaluate import Predictor, load_model
    model, cfg, _ = load_model(RUNS / r, DEVICE)
    predictor = Predictor(model, DEVICE, int(cfg["data"]["img_size"]))
    counts = {t: [] for t in THRESHOLDS}
    for start in range(0, len(VAL_DF), 16):
        chunk = VAL_DF.iloc[start:start + 16]
        pairs = [load_pair(DATA_ROOT, row) for _, row in chunk.iterrows()]
        for (_, gt), prob in zip(pairs, predictor([im for im, _ in pairs])):
            for t in THRESHOLDS:
                counts[t].append(confusion_counts((prob > t)[None], gt[None].astype(bool))[0])
    df = pd.DataFrame([{"run": r, "threshold": t, **validation_summary(np.array(counts[t]))} for t in THRESHOLDS])
    df.to_csv(cache, index=False)
    sweeps.append(df)
    del model, predictor
    if DEVICE.type == "mps":
        torch.mps.empty_cache()

SWEEP = pd.concat(sweeps, ignore_index=True) if sweeps else pd.DataFrame()
if len(SWEEP):
    fig, axes = plt.subplots(1, 3, figsize=(18, 4.2))
    for r, g in SWEEP.groupby("run"):
        axes[0].plot(g["threshold"], g["dice_tumor"], marker=".", color=COLORS[r], label=label(r))
        axes[1].plot(g["threshold"], 100 * g["healthy_fp_rate"], marker=".", color=COLORS[r], label=label(r))
        axes[2].plot(g["threshold"], g["dice"], marker=".", color=COLORS[r], label=label(r))
        best = g.loc[g["dice"].idxmax()]
        axes[2].scatter(best["threshold"], best["dice"], s=90, marker="*", color=COLORS[r], zorder=5)
    axes[0].set_title("Validation Dice — tumour slices")
    axes[1].set_title("Healthy validation slices with any FP pixel (%)")
    axes[2].set_title("Selection criterion: Dice over tumour + healthy (★ = best)")
    for ax in axes:
        ax.axvline(0.5, color="grey", ls=":"); ax.set_xlabel("probability threshold"); ax.legend(fontsize=7)
    plt.tight_layout(); savefig("threshold_sweep"); plt.show()

    TUNED = SWEEP.loc[SWEEP.groupby("run")["dice"].idxmax(), ["run", "threshold", "dice", "dice_tumor", "healthy_fp_rate"]]
    at_default = SWEEP[np.isclose(SWEEP["threshold"], 0.5)].set_index("run")
    TUNED["dice @0.5"] = TUNED["run"].map(at_default["dice"])
    display(TUNED.set_index("run"))

In [ ]:
TUNED_ROWS = []
if len(SWEEP):
    for _, row in TUNED.iterrows():
        r, thr = row["run"], float(row["threshold"])
        name = f"eval_thr{thr:.2f}"
        if np.isclose(thr, 0.5):
            print(f"{label(r)}: tuned threshold is the default 0.5 — no re-evaluation needed")
            continue
        if not (is_evaluated(r, name) and SKIP_COMPLETED) and RUN_TUNED_THRESHOLD_EVAL:
            run(["-m", "src.evaluate", "--run", f"{RUNS.relative_to(ROOT)}/{r}",
                 "--threshold", f"{thr:.2f}", "--out-name", name])
        if not is_evaluated(r, name):
            continue
        m_default, m_tuned = EVAL[r]["metrics"], load_eval(r, name)["metrics"]
        for tag, m in [("0.50 (default)", m_default), (f"{thr:.2f} (tuned on val)", m_tuned)]:
            TUNED_ROWS.append({"model": label(r), "threshold": tag, "test Dice": m["overall"]["dice"],
                               "precision": m["overall"]["precision"], "recall": m["overall"]["recall"],
                               "healthy FP ≥ 50 px": m["healthy"].get("fp_rate_largest_ge_50px", np.nan)})
    if TUNED_ROWS:
        display(pd.DataFrame(TUNED_ROWS).set_index(["model", "threshold"]).style
                .format("{:.3f}").format("{:.1%}", subset=["healthy FP ≥ 50 px"]))

---
## Step 11 · Qualitative results: successes and failure cases

Galleries written by `src.evaluate` (columns: slice · expert mask · predicted probability · error map with
**green** = correct tumour pixels, **red** = false positive, **blue** = missed tumour).

In [ ]:
for r in dict.fromkeys([BEST_RUN, BASELINE_RUN, HEALTHY_RUN]):
    if r not in EVAL:
        continue
    display(Markdown(f"#### {label(r)}"))
    for g in ["gallery_best", "gallery_typical", "gallery_worst", "gallery_healthy_fp"]:
        path = RUNS / r / "eval" / f"{g}.png"
        if path.exists():
            display(Image(filename=str(path), width=900))

### 11.1 Hardest cases

The test slices with the lowest Dice (averaged over all available models), plus the healthy slices with the
largest false positives, shown with the error map of every available model.

In [ ]:
if RUN_CROSS_MODEL_GALLERY and len(EVAL) >= 1:
    from src.evaluate import Predictor, load_model, postprocess

    dice_wide = pd.concat({r: e["test"].set_index("stem")["dice"] for r, e in EVAL.items()}, axis=1)
    hard = dice_wide.mean(axis=1).sort_values().head(4).index.tolist()
    lookup = splits["test"].set_index("stem")
    cases = [(s, lookup.loc[s]) for s in hard]
    base_h = EVAL.get(BASELINE_RUN, next(iter(EVAL.values())))["healthy"]
    if len(base_h):
        h_lookup = splits["healthy_test"].set_index("stem")
        for s in base_h.sort_values("largest_component_px", ascending=False)["stem"].head(2):
            cases.append((s, h_lookup.loc[s]))
    loaded = [(s, *load_pair(DATA_ROOT, row), row["tumor_type"]) for s, row in cases]

    preds = {}
    for r in EVAL:
        model, cfg, _ = load_model(RUNS / r, DEVICE)
        predictor = Predictor(model, DEVICE, int(cfg["data"]["img_size"]))
        preds[r] = [postprocess(p, 0.5, 0) for p in predictor([im for _, im, _, _ in loaded])]
        del model, predictor
    if DEVICE.type == "mps":
        torch.mps.empty_cache()

    fig, axes = plt.subplots(len(loaded), len(EVAL) + 1, figsize=(3.2 * (len(EVAL) + 1), 3.2 * len(loaded)), squeeze=False)
    for i, (stem, image, gt, ttype) in enumerate(loaded):
        axes[i, 0].imshow(fill_overlay(image, gt, (255, 200, 0)))
        axes[i, 0].set_title(f"{ttype} · expert mask", fontsize=9)
        for j, r in enumerate(EVAL, start=1):
            p = preds[r][i]
            c = confusion_counts(p[None], gt[None].astype(bool))[0]
            txt = f"Dice {summarize(c[None])['dice']:.2f}" if gt.any() else f"FP {int(c[1])} px"
            axes[i, j].imshow(error_overlay(image, p, gt))
            axes[i, j].set_title(f"{label(r)}\n{txt}", fontsize=8)
    for ax in axes.flat:
        ax.axis("off")
    plt.suptitle("Hardest test slices and worst healthy false positives — green TP · red FP · blue FN", y=1.0)
    plt.tight_layout(); savefig("cross_model_hard_cases"); plt.show()

### 11.2 Failure analysis

The quantitative evidence (Steps 7–9) points to four failure modes, which the galleries illustrate:

1. **Gliomas** — lowest Dice with both low precision and low recall: diffuse, infiltrative margins make the
   boundary ambiguous; most gliomas are fine (high median) but a minority fail badly (partial or complete misses).
2. **Small tumours** — precision drops: a few extra pixels weigh heavily against a small true area, and the
   512 → 256 px downsampling blurs small lesions.
3. **Pituitary tumours** — recall well above precision: the region tends to extend into neighbouring enhancing
   structures around the sella (over-segmentation, red in the error maps).
4. **Healthy slices** — a "tumour" is predicted on most healthy slices, because the model only ever saw slices
   containing a tumour.

When reading the galleries, check whether a failure is a complete miss (all blue), a wrong structure (all red)
or a boundary disagreement (green core with red/blue rims), and whether any expert mask looks questionable.

---
## Step 12 · Deployment and optimisation

1. **ONNX export** (`src.export_onnx`): the network plus a sigmoid is exported (opset 18) and compared with
   PyTorch on 50 test slices (probability difference and mask agreement).
2. **Static INT8 quantization** (`src.quantize_onnx`): QDQ format, per-channel INT8 weights, UINT8 activations,
   min-max calibration on 120 training slices (never test data).
3. **Benchmark** (`src.benchmark_deploy`): accuracy, healthy false positives, latency and file size for
   PyTorch (reference), ONNX FP32 (CPU and Core ML) and ONNX INT8 (CPU), on the full test set.
4. **Application**: `app/app.py` (Gradio) and `app/predict_cli.py` use the same ONNX pipeline (`src/inference.py`).

In [ ]:
FP32 = ROOT / "models" / f"{BEST_RUN or BASELINE_RUN}_fp32.onnx"
INT8 = ROOT / "models" / f"{BEST_RUN or BASELINE_RUN}_int8.onnx"
BENCH = ASSETS / "deployment_benchmark.json"

if RUN_DEPLOYMENT and BEST_RUN:
    if not (FP32.exists() and SKIP_COMPLETED):
        run(["-m", "src.export_onnx", "--run", f"{RUNS.relative_to(ROOT)}/{BEST_RUN}"])
    if not (INT8.exists() and SKIP_COMPLETED):
        run(["-m", "src.quantize_onnx", "--model", str(FP32.relative_to(ROOT))])
    if not (BENCH.exists() and SKIP_COMPLETED):
        run(["-m", "src.benchmark_deploy", "--models", str(FP32.relative_to(ROOT)), str(INT8.relative_to(ROOT)),
             "--reference-run", f"{RUNS.relative_to(ROOT)}/{BEST_RUN}", "--coreml"])
    if not any((ROOT / "app" / "examples").glob("*.jpg")):
        run(["scripts/prepare_app_examples.py"])

card_path = ROOT / "models" / "model_card.json"
if card_path.exists():
    card = json.loads(card_path.read_text())
    pc = card.get("parity_check", {})
    if pc:
        check("ONNX FP32 matches PyTorch (max prob. diff < 1e-3)", pc["max_abs_prob_diff"] < 1e-3,
              f"{pc['max_abs_prob_diff']:.2e}, mask agreement Dice {pc['mask_agreement_dice']:.4f}")
    print(json.dumps({k: card[k] for k in ("input", "output", "threshold", "quantization") if k in card}, indent=2))

if BENCH.exists():
    bench = json.loads(BENCH.read_text())
    DEPLOY = pd.DataFrame(bench["results"]).drop(columns=["by_tumor_type"]).set_index("variant")
    display(DEPLOY.style.format("{:.3f}").format("{:.1f}", subset=["size_mb", "model_ms_per_image", "end_to_end_ms_per_image"])
            .format("{:.1%}", subset=["healthy_fp_rate_ge_50px"]))
    onnx_rows = DEPLOY[DEPLOY.index.str.contains("ONNX")]
    ref_rows = DEPLOY[~DEPLOY.index.str.contains("ONNX")]
    if len(ref_rows) and len(onnx_rows):
        drop = (ref_rows["test_dice"].iloc[0] - onnx_rows["test_dice"]).abs().max()
        check("quantized / exported models keep test Dice within 0.01 of PyTorch", drop < 0.01, f"max |ΔDice| {drop:.4f}")

    fig, axes = plt.subplots(1, 3, figsize=(17, 4))
    names = [n.replace(" (", "\n(") for n in DEPLOY.index]
    for ax, col, title in zip(axes, ["size_mb", "model_ms_per_image", "test_dice"],
                              ["Model file size (MB)", "Model time per slice (ms)", "Test Dice"]):
        bars = ax.bar(names, DEPLOY[col], color=[f"C{i}" for i in range(len(DEPLOY))])
        ax.bar_label(bars, fmt="%.3f" if col == "test_dice" else "%.1f", fontsize=8)
        ax.set_title(title); ax.tick_params(axis="x", labelsize=7)
    axes[2].set_ylim(max(0, DEPLOY["test_dice"].min() - 0.05), min(1, DEPLOY["test_dice"].max() + 0.02))
    plt.tight_layout(); savefig("deployment_benchmark"); plt.show()
else:
    print("No deployment benchmark yet (RUN_DEPLOYMENT = False or export failed).")

In [ ]:
# Demo: the deployable pipeline (no PyTorch) on the example slices
from src.inference import OnnxSegmenter, render_overlay, render_heatmap

examples = sorted((ROOT / "app" / "examples").glob("*.jpg"))
model_file = INT8 if INT8.exists() else FP32
if examples and model_file.exists():
    seg = OnnxSegmenter(model_file)
    fig, axes = plt.subplots(2, len(examples), figsize=(3.3 * len(examples), 7), squeeze=False)
    rows = []
    for j, path in enumerate(examples):
        image = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
        res = seg.segment(image)
        verdict = "tumour detected" if res.stats["tumour_detected"] else "no tumour detected"
        axes[0, j].imshow(render_overlay(image, res))
        axes[0, j].set_title(f"{path.stem}\n{verdict} · {res.stats['tumour_area_pct']:.2f} %", fontsize=9)
        axes[1, j].imshow(render_heatmap(image, res.prob))
        rows.append({"example": path.stem, **{k: res.stats[k] for k in ("tumour_detected", "tumour_area_pct", "n_regions", "max_probability")}})
    for ax in axes.flat:
        ax.axis("off")
    axes[1, 0].text(-0.05, 0.5, "probability map", transform=axes[1, 0].transAxes, rotation=90, va="center", ha="right")
    plt.suptitle(f"Deployed pipeline ({model_file.name}) on the demo examples")
    plt.tight_layout(); savefig("app_examples"); plt.show()
    display(pd.DataFrame(rows))
    print("Start the web app in Terminal with:  python app/app.py   → http://127.0.0.1:7860")
    print("Save a screenshot of it as assets/app_screenshot.png for the README.")
else:
    print("No ONNX model or examples yet — run Step 12 first.")

---
## Step 13 · Report, summary of findings and validation report

`scripts/make_diagrams.py` draws the workflow, inference-pipeline and architecture diagrams;
`src.report` collects every result file and rewrites the *Results & Evaluation* block of `README.md`.

In [ ]:
if RUN_REPORT:
    run(["scripts/make_diagrams.py"])
    if BEST_RUN:
        run(["-m", "src.report", "--run", f"{RUNS.relative_to(ROOT)}/{BEST_RUN}"])
for name in ["workflow_diagram", "inference_pipeline", "model_architecture"]:
    if (ASSETS / f"{name}.png").exists():
        display(Image(filename=str(ASSETS / f"{name}.png"), width=1000))

In [ ]:
lines = []
if EVAL:
    b = EVAL[BEST_RUN]["metrics"]
    types = b["by_tumor_type"]
    hardest = min(types, key=lambda k: types[k]["dice"]); easiest = max(types, key=lambda k: types[k]["dice"])
    sizes = b["by_tumor_size"]
    lines += [
        f"* **Best model (test Dice):** {label(BEST_RUN)} — mean Dice **{b['overall']['dice']:.3f}** "
        f"(median {b['overall']['dice_median']:.3f}), IoU {b['overall']['iou']:.3f}, "
        f"{b['speed']['ms_per_image_forward']:.1f} ms/slice on {b['speed']['device']}.",
        f"* **Hardest tumour type:** {hardest} (Dice {types[hardest]['dice']:.3f}) vs easiest {easiest} "
        f"({types[easiest]['dice']:.3f}).",
        f"* **Tumour size matters:** small {sizes['small']['dice']:.3f} → large {sizes['large']['dice']:.3f} Dice.",
    ]
    if BASELINE_RUN in HEALTHY_EVAL and HEALTHY_RUN in HEALTHY_EVAL:
        fb = EVAL[BASELINE_RUN]["metrics"]["healthy"]["fp_rate_largest_ge_50px"]
        fh = EVAL[HEALTHY_RUN]["metrics"]["healthy"]["fp_rate_largest_ge_50px"]
        db = EVAL[BASELINE_RUN]["metrics"]["overall"]["dice"]
        dh = EVAL[HEALTHY_RUN]["metrics"]["overall"]["dice"]
        lines.append(f"* **Training with healthy slices:** healthy false-positive rate (≥ 50 px) "
                     f"{fb:.1%} → **{fh:.1%}**, tumour Dice {db:.3f} → {dh:.3f}.")
    if "unet_scratch" in EVAL and BASELINE_RUN in EVAL:
        lines.append(f"* **Transfer learning:** scratch U-Net {EVAL['unet_scratch']['metrics']['overall']['dice']:.3f} "
                     f"vs pretrained ResNet34 encoder {EVAL[BASELINE_RUN]['metrics']['overall']['dice']:.3f} test Dice.")
display(Markdown("\n".join(lines) if lines else "_No evaluated runs yet._"))

CHECK_TABLE = pd.DataFrame(CHECKS)
n_pass = int(CHECK_TABLE["passed"].sum()) if len(CHECK_TABLE) else 0
display(Markdown(f"**Validation checks passed: {n_pass} / {len(CHECK_TABLE)}**"))
display(CHECK_TABLE.style.apply(lambda s: ["background-color: #c6efce" if v else "background-color: #ffc7ce"
                                           for v in s], subset=["passed"]))

export = {
    "best_run": BEST_RUN,
    "results": RESULTS.reset_index().to_dict(orient="records") if len(RESULTS) else [],
    "statistics": STATS.reset_index().to_dict(orient="records") if len(STATS) else [],
    "postprocessing": POSTPROC_TABLE.to_dict(orient="records") if len(POSTPROC_TABLE) else [],
    "tuned_thresholds": TUNED.to_dict(orient="records") if len(SWEEP) else [],
    "deployment": json.loads(BENCH.read_text())["results"] if BENCH.exists() else [],
    "validation_checks": CHECKS,
}
(ASSETS / "results_summary.json").write_text(json.dumps(export, indent=2, default=float))
if len(RESULTS):
    RESULTS.to_csv(ASSETS / "results_table.csv")
print("Saved assets/results_summary.json, assets/results_table.csv and analysis_*.png figures.")
print("\nSubmission checklist:")
for item, ok in [
    ("README.md results block generated", "RESULTS:START" in (ROOT / "README.md").read_text() and (ASSETS / "RESULTS.md").exists()),
    ("workflow diagram", (ASSETS / "workflow_diagram.png").exists()),
    ("success & failure galleries", (ASSETS / "gallery_best.png").exists() and (ASSETS / "gallery_worst.png").exists()),
    ("INT8 ONNX model in models/", INT8.exists()),
    ("deployment benchmark", BENCH.exists()),
    ("app examples", any((ROOT / "app" / "examples").glob("*.jpg"))),
    ("app screenshot (manual)", (ASSETS / "app_screenshot.png").exists()),
    ("repository link filled in README", "<your-username>" not in (ROOT / "README.md").read_text()),
]:
    print(("  ✅ " if ok else "  ☐ ") + item)

**Before pushing to GitHub:** save this notebook *with outputs* (it is part of the submission), check the
checklist above, and fill in the repository links in section 9 of the README.